In [1]:
import pandas as pd
import unicodedata
import re
from difflib import SequenceMatcher
from itertools import combinations
from pathlib import Path

In [2]:
#Visualización de las primeras filas 

DATA_PATH = Path('/Users/valemoravale/Documents/UNIVERSIDAD /Semestre 5/ETL/workshop3/data/2017.csv')
df = pd.read_csv(DATA_PATH)
df.head()

,Country,Happiness.Rank,Happiness.Score,Whisker.high,Whisker.low,Economy..GDP.per.Capita.,Family,Health..Life.Expectancy.,Freedom,Generosity,Trust..Government.Corruption.,Dystopia.Residual
0,Norway,1,7.537,7.594445,7.479556,1.616463,1.533524,0.796667,0.635423,0.362012,0.315964,2.277027
1,Denmark,2,7.522,7.581728,7.462272,1.482383,1.551122,0.792566,0.626007,0.355280,0.400770,2.313707
2,Iceland,3,7.504,7.622030,7.385970,1.480633,1.610574,0.833552,0.627163,0.475540,0.153527,2.322715
3,Switzerland,4,7.494,7.561772,7.426227,1.564980,1.516912,0.858131,0.620071,0.290549,0.367007,2.276716
4,Finland,5,7.469,7.527542,7.410458,1.443572,1.540247,0.809158,0.617951,0.245483,0.382612,2.430182


In [3]:
#Numero de filas, columnas y tipo de cada una
print('Rows:', len(df))
print('Columns:', df.shape[1])
df.info()

Rows: 155
Columns: 12
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Country                        155 non-null    object 
 1   Happiness.Rank                 155 non-null    int64  
 2   Happiness.Score                155 non-null    float64
 3   Whisker.high                   155 non-null    float64
 4   Whisker.low                    155 non-null    float64
 5   Economy..GDP.per.Capita.       155 non-null    float64
 6   Family                         155 non-null    float64
 7   Health..Life.Expectancy.       155 non-null    float64
 8   Freedom                        155 non-null    float64
 9   Generosity                     155 non-null    float64
 10  Trust..Government.Corruption.  155 non-null    float64
 11  Dystopia.Residual              155 non-null    float64
dtypes: float64(10), int64(1), ob

In [4]:
#Valores faltantes y su porcentaje 
missing = df.isna().sum().to_frame('missing_count')
missing['missing_percent'] = (missing['missing_count'] / len(df) * 100).round(2)
missing

,missing_count,missing_percent
Country,0,0.0
Happiness.Rank,0,0.0
Happiness.Score,0,0.0
Whisker.high,0,0.0
Whisker.low,0,0.0
Economy..GDP.per.Capita.,0,0.0
Family,0,0.0
Health..Life.Expectancy.,0,0.0
Freedom,0,0.0
Generosity,0,0.0


In [5]:
#Columnas duplicadas y Country duplicado 
print('Full duplicated rows:', df.duplicated().sum())
print('Duplicated Country:', df['Country'].duplicated().sum())

Full duplicated rows: 0
Duplicated Country: 0


In [6]:
# Crear función para limpiar nombres
def limpiar_texto(texto):
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"[^a-z0-9\s]", "", texto)
    texto = re.sub(r"\s+", " ", texto)
    return texto

# Crear columna limpia
df["country_clean"] = df["Country"].apply(limpiar_texto)

# Buscar duplicados exactos después de limpiar
duplicados_limpios = df[df.duplicated("country_clean", keep=False)]

print("Duplicados después de limpiar:")
print(duplicados_limpios[["Country", "country_clean"]])

Duplicados después de limpiar:
Empty DataFrame
Columns: [Country, country_clean]
Index: []


In [7]:
countries = df["Country"].dropna().unique()

posibles_duplicados = []

for pais1, pais2 in combinations(countries, 2):
    pais1_clean = limpiar_texto(pais1)
    pais2_clean = limpiar_texto(pais2)

    similitud = SequenceMatcher(None, pais1_clean, pais2_clean).ratio()

    if similitud >= 0.85:
        posibles_duplicados.append({
            "Country 1": pais1,
            "Country 2": pais2,
            "Similitud": round(similitud, 3)
        })

posibles_duplicados_df = pd.DataFrame(posibles_duplicados)

print(posibles_duplicados_df)

   Country 1 Country 2  Similitud
0    Iceland   Ireland      0.857
1  Australia   Austria      0.875


In [10]:
# Seleccionar solo columnas numéricas
numeric_cols = df.select_dtypes(include=["number"]).columns

ceros = (df[numeric_cols] == 0).sum()
negativos = (df[numeric_cols] < 0).sum()

print("Ceros por columna:")
print(ceros)
print("-------------------------------------")
print("Negativos por columna:")
print(negativos)

Ceros por columna:
Happiness.Rank                   0
Happiness.Score                  0
Whisker.high                     0
Whisker.low                      0
Economy..GDP.per.Capita.         1
Family                           1
Health..Life.Expectancy.         1
Freedom                          1
Generosity                       1
Trust..Government.Corruption.    1
Dystopia.Residual                0
dtype: int64
-------------------------------------
Negativos por columna:
Happiness.Rank                   0
Happiness.Score                  0
Whisker.high                     0
Whisker.low                      0
Economy..GDP.per.Capita.         0
Family                           0
Health..Life.Expectancy.         0
Freedom                          0
Generosity                       0
Trust..Government.Corruption.    0
Dystopia.Residual                0
dtype: int64


In [9]:
filas_no_validas = df[df[numeric_cols].le(0).any(axis=1)]

print(filas_no_validas)

                      Country  Happiness.Rank  Happiness.Score  Whisker.high  \
86                     Greece              87            5.227      5.325246   
89     Bosnia and Herzegovina              90            5.182      5.276336   
138                   Lesotho             139            3.808      4.044344   
139                    Angola             140            3.795      3.951642   
154  Central African Republic             155            2.693      2.864884   

     Whisker.low  Economy..GDP.per.Capita.    Family  \
86      5.128754                  1.289487  1.239415   
89      5.087665                  0.982409  1.069336   
138     3.571656                  0.521021  1.190095   
139     3.638358                  0.858428  1.104412   
154     2.521116                  0.000000  0.000000   

     Health..Life.Expectancy.   Freedom  Generosity  \
86                   0.810199  0.095731    0.000000   
89                   0.705186  0.204403    0.328867   
138              

In [11]:
#ydata-profiling
from ydata_profiling import ProfileReport
profile = ProfileReport (df, title="Raw sale data profiling report", explorative= True)
profile.to_file("raw_sale_data_data_profile2017.html" )

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 243.49it/s]
